# Pytorch tutorials 

## 4 - Build the Neural Network  

Libs

In [2]:
import os # for file handling
import torch # for deep learning
from torch import nn # for neural network layers
from torch.utils.data import DataLoader # for managing data
from torchvision import datasets, transforms # for computer vision datasets

### Get device for training 

In [3]:
device = (
    "cuda"
    if torch.cuda.is_available() # check if cuda is available
    else "mps"
    if torch.backends.mps.is_available() # check if mps is available
    else "cpu"
)

print(f"Using {device} device")

Using cpu device


### Define the class

In [5]:
class NeuralNetwork(nn.Module): # create a neural network class
    def __init__(self): # initialize the class
        super().__init__() # call the parent class constructor
        self.flatten = nn.Flatten() # flatten the input
        self.linear_relu_stack = nn.Sequential( # create a sequence of layers
            nn.Linear(28*28, 512), # input layer (input of size 28*28, output of size 512)
            nn.ReLU(), # activation function
            nn.Linear(512, 512), # hidden layer (input of size 512, output of size 512)
            nn.ReLU(), 
            nn.Linear(512, 10), # output layer (input of size 512, output of size 10)
        )

    def forward(self, x): # forward pass
        x = self.flatten(x) # flatten the input
        logits = self.linear_relu_stack(x) # pass the input through the layers (linear_relu_stack of the initialization)
        return logits

Create an instance and move it to the deivice 

In [6]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Example of the model usage 

In [11]:
X = torch.rand(1, 28, 28, device=device) # create a random tensor of size 1x28x28

# Forward pass
logits = model(X) # pass the tensor through the model
pred_probab = nn.Softmax(dim=1)(logits) # apply softmax to the output
y_pred = pred_probab.argmax(1) # get the predicted class

print(f"Predicted class: {y_pred}")

Predicted class: tensor([4])


### Model layers

A single input

In [12]:
input_image = torch.rand(3,28,28)
print(input_image.size()) # a rbg quantity for each pixel of the picture 28x28

torch.Size([3, 28, 28])


### nn.Flatten
Flat an image. 
Simply $3 \times 28 \times 28 \longrightarrow 3 \times 784 $

In [13]:
flatten = nn.Flatten() # create a flatten layer 

flat_image = flatten(input_image) # flatten the input
print(flat_image.size())

torch.Size([3, 784])


### nn.Linear 
A linear trasformation 

In [14]:
layer1 = nn.Linear(in_features=28*28, out_features=20) # create a linear layer
hidden1 = layer1(flat_image) # pass the input through the layer

print(hidden1.size())

torch.Size([3, 20])


### nn.ReLu
A non linear mapping after a linear trasformation.
Simply $x \longrightarrow x $  if $x >0$ AND $x \longrightarrow 0 $ otherwise

In [15]:
print(f"Before ReLU: {hidden1}\n\n")

hidden1 = nn.ReLU()(hidden1) # apply the ReLU activation function

print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.0492,  0.1973,  0.2142, -0.0792, -0.1188,  0.7051, -0.3509,  0.1384,
         -0.2989,  0.0839, -0.2991, -0.4734, -0.4832, -0.1630, -0.0154,  0.0029,
         -0.4572,  0.3929, -0.2103,  0.6796],
        [-0.0194,  0.2273,  0.2077, -0.0803,  0.0941,  0.4566, -0.3286, -0.0518,
         -0.2353, -0.0593, -0.0747, -0.8809, -0.2842,  0.1150, -0.1075, -0.3503,
         -0.2656,  0.5171, -0.2821,  0.4408],
        [ 0.0410,  0.5090,  0.2355, -0.2330, -0.0658,  0.0999, -0.3004,  0.2479,
         -0.0207,  0.0273,  0.0261, -0.1795, -0.3381, -0.1545,  0.1261, -0.2862,
         -0.3516,  0.4261,  0.2585,  0.4518]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0492, 0.1973, 0.2142, 0.0000, 0.0000, 0.7051, 0.0000, 0.1384, 0.0000,
         0.0839, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0029, 0.0000, 0.3929,
         0.0000, 0.6796],
        [0.0000, 0.2273, 0.2077, 0.0000, 0.0941, 0.4566, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.1150, 0.00

### nn.Sequential
A ordered container of modules. Each module is applyied following the specified order. 

In [16]:
seq_modules = nn.Sequential( # create a sequence of layers
    flatten, # flatten the input
    layer1, # linear layer
    nn.ReLU(), # activation function
    nn.Linear(20, 10) # output layer
)

# Apply the sequence of layers to an image
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

### nn.Softmax
The logits output has value in $[-\infty, \infty]$. Therefore, we use the softmax in order to make them in $[0,1]$. So that, we can see each value as the probability of the respective class. 

`Dim = 1` means that the sum of the values of the softmax on the first dimension (dim = 1) must be equal to 1.   

In [17]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

### Model parameters 

In [18]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n") # param[:2] means the first two element of the param tensor

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0272, -0.0342, -0.0123,  ..., -0.0019,  0.0175,  0.0156],
        [ 0.0344,  0.0288, -0.0170,  ..., -0.0346, -0.0125, -0.0203]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([ 0.0282, -0.0145], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[-0.0126, -0.0421, -0.0097,  ..., -0.0367, -0.0253,  0.0199],
        [-0.0235, -0.0258, -0.0427,  ..., -0.0348, -0.0257, -0.0225]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | 